In [1]:
import sys, pathlib
repo_root = pathlib.Path("/Users/muhammadnumanmuttaqi/Documents/MScITBE/Thesis/Thesis/Geometry-to-Ontology")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [4]:
from rdflib import Graph
import pathlib

# Paths
core = pathlib.Path('../ontology/core/Resplan.ttl')
input_plan = pathlib.Path('../output/imp_resplan_ttl/plan_00000_drop_door.ttl')
construct_dir = pathlib.Path('../ontology/construct')
construct_files = [
    construct_dir / 'DoorConstruct.rq',
    construct_dir / 'InteriorWallConstruct.rq',
    construct_dir / 'WindowConstruct.rq',
]
output_path = pathlib.Path('../output/inferred_resplan_ttl/plan_00000_constructed.ttl')

# Merge data graph(s)
g = Graph()
for src in (core, input_plan):
    print(f'Loading: {src}')
    g.parse(src, format='turtle')
print(f'Loaded triples: {len(g)}')

# Apply CONSTRUCT queries (use g.query, not g.update)
for cq in construct_files:
    text = cq.read_text()
    print(f'Applying construct: {cq}')
    res = g.query(text)
    try:
        g += res.graph  # add constructed triples
    except Exception:
        # If result is not a Graph (older rdflib), iterate bindings
        for t in res:
            g.add(t)
print(f'Triples after construct: {len(g)}')

# Save enriched graph
output_path.parent.mkdir(parents=True, exist_ok=True)
g.serialize(output_path, format='turtle')
print(f'Saved: {output_path}')


Loading: ../ontology/core/Resplan.ttl
Loading: ../output/imp_resplan_ttl/plan_00000_drop_door.ttl
Loaded triples: 1120
Applying construct: ../ontology/construct/DoorConstruct.rq
Applying construct: ../ontology/construct/InteriorWallConstruct.rq
Applying construct: ../ontology/construct/WindowConstruct.rq
Triples after construct: 1128
Saved: ../output/inferred_resplan_ttl/plan_00000_constructed.ttl


In [5]:
# DOOR RULES MISSING CHECKER
from pyshacl import validate
from rdflib import RDF, Graph, Namespace, RDFS, BNode, Literal

def _short(node):
    return str(node).split("#")[-1] if "#" in str(node) else str(node)

data_graph = Graph().parse("../output/inferred_resplan_ttl/plan_00000_constructed.ttl", format="turtle")
shacl_graph = Graph().parse("../ontology/rules/DoorRule.shacl.ttl", format="turtle")
onto_graph = Graph().parse("../ontology/core/Resplan.ttl", format="turtle")

conforms, report_graph, _ = validate(
    data_graph,
    shacl_graph=shacl_graph,
    ont_graph=onto_graph,
    inference="rdfs",
    meta_shacl=True,
    advanced=True,
)

sh = Namespace("http://www.w3.org/ns/shacl#")

if conforms:
    print("All door rules satisfied.")
else:
    print("Door rule violations found:")

    handled_sparql_rules = set()

    for result in report_graph.subjects(RDF.type, sh.ValidationResult):
        raw_shape = report_graph.value(result, sh.sourceShape)

        # mapped from PropertyShape/SPARQL bnode to main NodeShape (using shacl_graph!)
        rule_node = raw_shape
        if isinstance(raw_shape, BNode):
            for ns in shacl_graph.subjects(sh.property, raw_shape):
                rule_node = ns
                break

        rule_id = _short(rule_node)
        rule_label = report_graph.value(rule_node, RDFS.label) or shacl_graph.value(rule_node, RDFS.label)
        rule_name = str(rule_label) if rule_label else rule_id

        comp = report_graph.value(result, sh.sourceConstraintComponent)
        focus = _short(report_graph.value(result, sh.focusNode))
        message_node = report_graph.value(result, sh.resultMessage)
        message = str(message_node) if isinstance(message_node, Literal) or message_node else ""

        if comp == sh.SPARQLConstraintComponent:
            if rule_node in handled_sparql_rules:
                continue  # already processed this rule
            handled_sparql_rules.add(rule_node)
            
            sc = report_graph.value(result, sh.sourceConstraint)
            sparql = report_graph.value(sc, sh.select)

            for row in data_graph.query(str(sparql)):
                if len(row) >= 2:
                    print(f"{_short(row[0])} – {_short(row[1])}: {message} ({rule_name})")
                else:
                    print(f"{' – '.join(_short(v) for v in row)}: {message} ({rule_name})")
        else:
            print(f"- {focus} violates {rule_name}: {message}")


Door rule violations found:
BED-01 – LIV-01: Door must connect exactly two spaces via connected_via_door relationships. (DW2: Missing door between spaces that has connectedViaDoor relationship)
a016f6bf-fc5a-46aa-a9ac-6b0d1a88fa9e: Door connects two spaces but their shared boundary wall is missing. (DW3: Host wall must bound both connected spaces)
b3bb3718-5083-4ebd-ab06-04766a76d805: Door connects two spaces but their shared boundary wall is missing. (DW3: Host wall must bound both connected spaces)
- b3bb3718-5083-4ebd-ab06-04766a76d805 violates DW1: Door must be hosted by a wall: This door's wall is missing
- a016f6bf-fc5a-46aa-a9ac-6b0d1a88fa9e violates DW1: Door must be hosted by a wall: This door's wall is missing
